#  四剖面三参数汇总图

## HCT-QHead

In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FormatStrFormatter, FuncFormatter, MaxNLocator


# ------------------------------------------------------------------------------
# 1. 路径与数据配置
# ------------------------------------------------------------------------------
def find_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "HCT-QHead" / "byprofile").is_dir() and (
            candidate / "section"
        ).is_dir():
            return candidate
    raise FileNotFoundError("未找到项目根目录 HCT-QHead_Synthesis_AVO。")


PROJECT_ROOT = find_project_root()
PRED_ROOT = PROJECT_ROOT / "HCT-QHead" / "byprofile"
SECTION_ROOT = PROJECT_ROOT / "section"
LOG_DATA_DIR = PROJECT_ROOT / "well"
OUTPUT_DIR = PROJECT_ROOT / "analysis" / "section"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PNG = OUTPUT_DIR / "HCT-QHead_profiles_s1-s4_Ip_Is_Rho_A4.png"
OUTPUT_PDF = OUTPUT_DIR / "HCT-QHead_profiles_s1-s4_Ip_Is_Rho_A4.pdf"
SAVE_PDF = True
SHOW_FIGURE = True

PROFILE_NAMES = ["profile_s1", "profile_s2", "profile_s3", "profile_s4"]
PROFILE_LABELS = {
    "profile_s1": "S1",
    "profile_s2": "S2",
    "profile_s3": "S3",
    "profile_s4": "S4",
}
TARGETS = ["Ip", "Is", "Dn"]
COLUMN_TITLES = {
    "Ip": "P-wave impedance",
    "Is": "S-wave impedance",
    "Dn": "Density",
}
PREDICTION_LABEL = "median"

# 每条剖面的独立时间范围；设为 None 使用完整时间范围。
PROFILE_TIME_RANGES = {
    "profile_s1": (2.0, 2.8),
    "profile_s2": (2.0, 2.8),
    "profile_s3": (2.0, 2.8),
    "profile_s4": (2.0, 2.8),
}

# ------------------------------------------------------------------------------
# 2. 色标接口
# ------------------------------------------------------------------------------
COLOR_PERCENTILES = (2.0, 98.0)

# None：合并四条剖面的有效值后，自动使用 COLOR_PERCENTILES。
# (vmin, vmax)：手动固定该列的色标范围。可只设置其中一列。
CMAP_RANGES = {
    "Ip": (5700,10700),
    "Is": (3500,6700),
    # "Dn": (2.19,2.62),
    "Dn": (2.21,2.6),
}

CMAP_NAME = "jet"
INTERPOLATION = "bilinear"

# ------------------------------------------------------------------------------
# 3. 版式接口
# ------------------------------------------------------------------------------
A4_LANDSCAPE_INCHES = (11.69, 8.27)
FONT_SIZE = 9  # 小五号
DISPLAY_DPI = 120  # Notebook 中显示分辨率
SAVE_DPI = 600  # PNG 输出分辨率
ROW_HSPACE = 0.6  # 四行剖面之间的间隔；数值越大，行间距越大
TIME_LABEL = "Time(s)"  # 每行纵轴的时间标题
TIME_LABEL_PAD = 2  # Time(s) 与时间刻度数字之间的距离
PROFILE_LABEL_ROTATION = 0  # S1-S4 水平正向显示
PROFILE_LABEL_X = -0.16  # S1-S4 的水平位置（axes 坐标，越小越靠左）

OVERLAY_TRUE_LOG = True  # 沿用参考代码，叠加真实测井彩条
SHOW_WELL_NAMES = True
LOG_COLOR_HALF_WIDTH = 0.8
LOG_BLACK_BORDER_WIDTH = 0.5  # 真实井彩条单侧黑边宽度，单位为 Trace
WELL_LABEL_Y_AXES = 1.015

mpl.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Noto Sans CJK JP", "DejaVu Sans"],
        "axes.unicode_minus": False,
        "font.size": FONT_SIZE,
        "axes.titlesize": FONT_SIZE,
        "axes.labelsize": FONT_SIZE,
        "xtick.labelsize": FONT_SIZE,
        "ytick.labelsize": FONT_SIZE,
    }
)

print(f"项目根目录：{PROJECT_ROOT}")
print(f"图片输出：{OUTPUT_PNG}")

# ------------------------------------------------------------------------------
# 4. 数据读取与辅助函数
# ------------------------------------------------------------------------------
def clean_columns(dataframe):
    dataframe = dataframe.copy()
    dataframe.columns = [
        str(column).replace("\ufeff", "").strip() for column in dataframe.columns
    ]
    return dataframe


def find_time_column(dataframe):
    for column in dataframe.columns:
        if str(column).lower().startswith("time"):
            return column
    raise ValueError(f"未找到时间列，当前列名：{list(dataframe.columns)}")


def read_prediction(csv_path, time_range):
    if not csv_path.exists():
        raise FileNotFoundError(f"预测文件不存在：{csv_path}")

    dataframe = clean_columns(pd.read_csv(csv_path))
    time_column = find_time_column(dataframe)
    time_axis = pd.to_numeric(dataframe[time_column], errors="coerce").to_numpy(
        dtype=float
    )
    data_columns = [column for column in dataframe.columns if column != time_column]
    matrix = (
        dataframe[data_columns]
        .apply(pd.to_numeric, errors="coerce")
        .to_numpy(dtype=float)
    )

    valid_time = np.isfinite(time_axis)
    if time_range is not None:
        time_start, time_end = time_range
        if time_start is not None:
            valid_time &= time_axis >= float(time_start)
        if time_end is not None:
            valid_time &= time_axis <= float(time_end)

    time_axis = time_axis[valid_time]
    matrix = matrix[valid_time]
    matrix[(matrix == 0.0) | ~np.isfinite(matrix)] = np.nan

    if time_axis.size == 0 or not np.isfinite(matrix).any():
        raise ValueError(f"指定范围内没有有效数据：{csv_path}")

    return time_axis, matrix


def read_well_locations(profile_name):
    csv_path = SECTION_ROOT / profile_name / "profile_wells_location.csv"
    if not csv_path.exists():
        return pd.DataFrame(columns=["wellname", "trace_index"])

    dataframe = clean_columns(pd.read_csv(csv_path))
    if not {"wellname", "trace_index"}.issubset(dataframe.columns):
        return pd.DataFrame(columns=["wellname", "trace_index"])

    dataframe["wellname"] = dataframe["wellname"].astype(str).str.strip()
    dataframe["trace_index"] = pd.to_numeric(dataframe["trace_index"], errors="coerce")
    dataframe = dataframe.loc[np.isfinite(dataframe["trace_index"])].copy()
    dataframe["trace_index"] = dataframe["trace_index"].astype(int)
    return dataframe.sort_values("trace_index").reset_index(drop=True)


def split_contiguous_indices(indices):
    indices = np.asarray(indices, dtype=int)
    if indices.size == 0:
        return []
    return np.split(indices, np.where(np.diff(indices) > 1)[0] + 1)


def read_well_log_segments(well_name, target, time_min, time_max):
    csv_path = LOG_DATA_DIR / f"{well_name}.csv"
    if not csv_path.exists():
        return []

    dataframe = clean_columns(pd.read_csv(csv_path))
    time_column = find_time_column(dataframe)
    if target not in dataframe.columns:
        return []

    log_time = pd.to_numeric(dataframe[time_column], errors="coerce").to_numpy(
        dtype=float
    )
    log_values = pd.to_numeric(dataframe[target], errors="coerce").to_numpy(dtype=float)
    valid = (
        np.isfinite(log_time)
        & np.isfinite(log_values)
        & (log_values != 0.0)
        & (log_time >= time_min)
        & (log_time <= time_max)
    )

    segments = []
    for group in split_contiguous_indices(np.where(valid)[0]):
        if group.size:
            order = np.argsort(log_time[group])
            segments.append((log_time[group][order], log_values[group][order]))
    return segments


def resolve_cmap_range(target, matrices):
    manual_range = CMAP_RANGES.get(target)
    if manual_range is not None:
        if len(manual_range) != 2:
            raise ValueError(f"CMAP_RANGES['{target}'] 必须为 None 或 (vmin, vmax)。")
        value_min, value_max = map(float, manual_range)
        if (
            not np.isfinite(value_min)
            or not np.isfinite(value_max)
            or value_min >= value_max
        ):
            raise ValueError(f"CMAP_RANGES['{target}'] 无效：{manual_range}")
        return value_min, value_max, "手动"

    finite_parts = [matrix[np.isfinite(matrix)] for matrix in matrices]
    finite_parts = [values for values in finite_parts if values.size]
    if not finite_parts:
        raise ValueError(f"{target} 在四条剖面中均无有效值。")

    all_values = np.concatenate(finite_parts)
    value_min, value_max = np.percentile(all_values, COLOR_PERCENTILES).astype(float)
    if value_max <= value_min:
        delta = max(abs(value_min), 1.0) * 1e-6
        value_min, value_max = value_min - delta, value_max + delta
    return value_min, value_max, f"{COLOR_PERCENTILES[0]:g}%–{COLOR_PERCENTILES[1]:g}%"


def draw_well_logs(ax, profile_name, target, matrix, time_axis, cmap, vmin, vmax):
    wells = read_well_locations(profile_name)
    time_min, time_max = float(np.min(time_axis)), float(np.max(time_axis))

    for _, row in wells.iterrows():
        trace_index = int(row["trace_index"])
        if not 0 <= trace_index < matrix.shape[1]:
            continue
        trace_x = trace_index + 0.5

        if OVERLAY_TRUE_LOG:
            for segment_time, segment_values in read_well_log_segments(
                row["wellname"], target, time_min, time_max
            ):
                if len(segment_time) < 2:
                    continue
                ax.fill_betweenx(
                    segment_time,
                    trace_x - LOG_COLOR_HALF_WIDTH - LOG_BLACK_BORDER_WIDTH,
                    trace_x + LOG_COLOR_HALF_WIDTH + LOG_BLACK_BORDER_WIDTH,
                    color="black",
                    edgecolor="none",
                    zorder=10,
                )
                x_edges = np.array(
                    [trace_x - LOG_COLOR_HALF_WIDTH, trace_x + LOG_COLOR_HALF_WIDTH]
                )
                cell_values = (
                    0.5 * (segment_values[:-1] + segment_values[1:])
                ).reshape(-1, 1)
                ax.pcolormesh(
                    x_edges,
                    segment_time,
                    cell_values,
                    shading="flat",
                    cmap=cmap,
                    vmin=vmin,
                    vmax=vmax,
                    zorder=11,
                )

        if SHOW_WELL_NAMES:
            ax.text(
                trace_x,
                WELL_LABEL_Y_AXES,
                str(row["wellname"]),
                transform=ax.get_xaxis_transform(),
                ha="center",
                va="bottom",
                fontsize=FONT_SIZE,
                clip_on=False,
                zorder=20,
            )


def format_colorbar(colorbar, target):
    if target in {"Ip", "Is"}:
        colorbar.ax.xaxis.set_major_formatter(
            FuncFormatter(lambda value, position: f"{value / 1000:.2f}")
        )
    else:
        colorbar.ax.xaxis.set_major_formatter(FormatStrFormatter("%.2f"))

    labels = {
        "Ip": r"P-impedance  $\times 10^6$ kg/(m$^2$ s)",
        "Is": r"S-impedance  $\times 10^6$ kg/(m$^2$ s)",
        "Dn": r"Density  $\times 10^3$ kg/m$^3$",
    }
    colorbar.set_label(labels[target], fontsize=FONT_SIZE, labelpad=3)
    colorbar.ax.tick_params(axis="x", labelsize=FONT_SIZE, pad=2)


# ------------------------------------------------------------------------------
# 5. 一次读取全部 12 个矩阵，并计算三列共享色限
# ------------------------------------------------------------------------------
plot_data = {}
for profile_name in PROFILE_NAMES:
    for target in TARGETS:
        csv_path = PRED_ROOT / profile_name / f"{target}_{PREDICTION_LABEL}.csv"
        plot_data[(profile_name, target)] = read_prediction(
            csv_path, PROFILE_TIME_RANGES.get(profile_name)
        )

column_ranges = {}
for target in TARGETS:
    matrices = [plot_data[(profile_name, target)][1] for profile_name in PROFILE_NAMES]
    column_ranges[target] = resolve_cmap_range(target, matrices)
    vmin, vmax, source = column_ranges[target]
    print(f"{target}: {source} 色标范围 = {vmin:.6g} ～ {vmax:.6g}")


# ------------------------------------------------------------------------------
# 6. 4 × 3 A4 横向绘图
# ------------------------------------------------------------------------------
figure = plt.figure(figsize=A4_LANDSCAPE_INCHES, dpi=DISPLAY_DPI)
grid = figure.add_gridspec(
    nrows=5,
    ncols=3,
    height_ratios=[1, 1, 1, 1, 0.09],
    left=0.065,
    right=0.99,
    top=0.935,
    bottom=0.09,
    hspace=ROW_HSPACE,
    wspace=0.10,
)

axes = np.empty((4, 3), dtype=object)
images = {}
cmap = plt.get_cmap(CMAP_NAME).copy()
cmap.set_bad(color="white")

for row_index, profile_name in enumerate(PROFILE_NAMES):
    for column_index, target in enumerate(TARGETS):
        ax = figure.add_subplot(grid[row_index, column_index])
        axes[row_index, column_index] = ax

        time_axis, matrix = plot_data[(profile_name, target)]
        vmin, vmax, _ = column_ranges[target]
        image = ax.imshow(
            matrix,
            aspect="auto",
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            extent=[
                0,
                matrix.shape[1],
                float(np.max(time_axis)),
                float(np.min(time_axis)),
            ],
            interpolation=INTERPOLATION,
            zorder=1,
        )
        images[target] = image

        draw_well_logs(ax, profile_name, target, matrix, time_axis, cmap, vmin, vmax)

        if row_index == 0:
            ax.set_title(COLUMN_TITLES[target], fontsize=FONT_SIZE, pad=14)

        ax.set_xlim(0, matrix.shape[1])
        ax.set_ylim(float(np.max(time_axis)), float(np.min(time_axis)))
        ax.set_xlabel("Trace", fontsize=FONT_SIZE, labelpad=2)
        ax.xaxis.set_major_locator(MaxNLocator(nbins=5, integer=True))
        ax.tick_params(axis="both", labelsize=FONT_SIZE, length=2.5, pad=2)

        if column_index == 0:
            ax.set_ylabel(TIME_LABEL, fontsize=FONT_SIZE, labelpad=TIME_LABEL_PAD)
            ax.text(
                PROFILE_LABEL_X,
                0.5,
                PROFILE_LABELS[profile_name],
                transform=ax.transAxes,
                fontsize=FONT_SIZE,
                rotation=PROFILE_LABEL_ROTATION,
                ha="right",
                va="center",
                clip_on=False,
            )
            ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
        else:
            ax.tick_params(axis="y", left=False, labelleft=False)

        for spine in ax.spines.values():
            spine.set_linewidth(0.6)

for column_index, target in enumerate(TARGETS):
    colorbar_ax = figure.add_subplot(grid[4, column_index])
    colorbar = figure.colorbar(
        images[target], cax=colorbar_ax, orientation="horizontal"
    )
    colorbar.locator = MaxNLocator(nbins=5)
    colorbar.update_ticks()
    format_colorbar(colorbar, target)

# 不使用 bbox_inches='tight'，确保输出画布保持严格的 A4 横向尺寸。
figure.savefig(OUTPUT_PNG, dpi=SAVE_DPI)
if SAVE_PDF:
    figure.savefig(OUTPUT_PDF)

print(f"已保存 PNG：{OUTPUT_PNG}")
if SAVE_PDF:
    print(f"已保存 PDF：{OUTPUT_PDF}")

if SHOW_FIGURE:
    plt.show()
else:
    plt.close(figure)

## HCT-PHead

In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FormatStrFormatter, FuncFormatter, MaxNLocator


# ------------------------------------------------------------------------------
# 1. 路径与数据配置
# ------------------------------------------------------------------------------
def find_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "HCT-PHead" / "byprofile").is_dir() and (
            candidate / "section"
        ).is_dir():
            return candidate
    raise FileNotFoundError("未找到项目根目录 HCT-QHead_Synthesis_AVO。")


PROJECT_ROOT = find_project_root()
PRED_ROOT = PROJECT_ROOT / "HCT-PHead" / "byprofile"
SECTION_ROOT = PROJECT_ROOT / "section"
LOG_DATA_DIR = PROJECT_ROOT / "well"
OUTPUT_DIR = PROJECT_ROOT / "analysis" / "section"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PNG = OUTPUT_DIR / "HCT-PHead_profiles_s1-s4_Ip_Is_Rho_A4.png"
OUTPUT_PDF = OUTPUT_DIR / "HCT-PHead_profiles_s1-s4_Ip_Is_Rho_A4.pdf"
SAVE_PDF = True
SHOW_FIGURE = True

PROFILE_NAMES = ["profile_s1", "profile_s2", "profile_s3", "profile_s4"]
PROFILE_LABELS = {
    "profile_s1": "S1",
    "profile_s2": "S2",
    "profile_s3": "S3",
    "profile_s4": "S4",
}
TARGETS = ["Ip", "Is", "Dn"]
COLUMN_TITLES = {
    "Ip": "P-wave impedance",
    "Is": "S-wave impedance",
    "Dn": "Density",
}
PREDICTION_LABEL = "PHead"

# 每条剖面的独立时间范围；设为 None 使用完整时间范围。
PROFILE_TIME_RANGES = {
    "profile_s1": (2.0, 2.8),
    "profile_s2": (2.0, 2.8),
    "profile_s3": (2.0, 2.8),
    "profile_s4": (2.0, 2.8),
}

# ------------------------------------------------------------------------------
# 2. 色标接口
# ------------------------------------------------------------------------------
COLOR_PERCENTILES = (2.0, 98.0)

# None：合并四条剖面的有效值后，自动使用 COLOR_PERCENTILES。
# (vmin, vmax)：手动固定该列的色标范围。可只设置其中一列。
CMAP_RANGES = {
    "Ip": (5700,10700),
    "Is": (3500,6700),
    "Dn": (2.21,2.60),
}


CMAP_NAME = "jet"
INTERPOLATION = "bilinear"

# ------------------------------------------------------------------------------
# 3. 版式接口
# ------------------------------------------------------------------------------
A4_LANDSCAPE_INCHES = (11.69, 8.27)
FONT_SIZE = 9  # 小五号
DISPLAY_DPI = 120  # Notebook 中显示分辨率
SAVE_DPI = 600  # PNG 输出分辨率
ROW_HSPACE = 0.6  # 四行剖面之间的间隔；数值越大，行间距越大
TIME_LABEL = "Time(s)"  # 每行纵轴的时间标题
TIME_LABEL_PAD = 2  # Time(s) 与时间刻度数字之间的距离
PROFILE_LABEL_ROTATION = 0  # S1-S4 水平正向显示
PROFILE_LABEL_X = -0.16  # S1-S4 的水平位置（axes 坐标，越小越靠左）

OVERLAY_TRUE_LOG = True  # 沿用参考代码，叠加真实测井彩条
SHOW_WELL_NAMES = True
LOG_COLOR_HALF_WIDTH = 0.8
LOG_BLACK_BORDER_WIDTH = 0.5  # 真实井彩条单侧黑边宽度，单位为 Trace
WELL_LABEL_Y_AXES = 1.015

mpl.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Noto Sans CJK JP", "DejaVu Sans"],
        "axes.unicode_minus": False,
        "font.size": FONT_SIZE,
        "axes.titlesize": FONT_SIZE,
        "axes.labelsize": FONT_SIZE,
        "xtick.labelsize": FONT_SIZE,
        "ytick.labelsize": FONT_SIZE,
    }
)

print(f"项目根目录：{PROJECT_ROOT}")
print(f"图片输出：{OUTPUT_PNG}")

# ------------------------------------------------------------------------------
# 4. 数据读取与辅助函数
# ------------------------------------------------------------------------------
def clean_columns(dataframe):
    dataframe = dataframe.copy()
    dataframe.columns = [
        str(column).replace("\ufeff", "").strip() for column in dataframe.columns
    ]
    return dataframe


def find_time_column(dataframe):
    for column in dataframe.columns:
        if str(column).lower().startswith("time"):
            return column
    raise ValueError(f"未找到时间列，当前列名：{list(dataframe.columns)}")


def read_prediction(csv_path, time_range):
    if not csv_path.exists():
        raise FileNotFoundError(f"预测文件不存在：{csv_path}")

    dataframe = clean_columns(pd.read_csv(csv_path))
    time_column = find_time_column(dataframe)
    time_axis = pd.to_numeric(dataframe[time_column], errors="coerce").to_numpy(
        dtype=float
    )
    data_columns = [column for column in dataframe.columns if column != time_column]
    matrix = (
        dataframe[data_columns]
        .apply(pd.to_numeric, errors="coerce")
        .to_numpy(dtype=float)
    )

    valid_time = np.isfinite(time_axis)
    if time_range is not None:
        time_start, time_end = time_range
        if time_start is not None:
            valid_time &= time_axis >= float(time_start)
        if time_end is not None:
            valid_time &= time_axis <= float(time_end)

    time_axis = time_axis[valid_time]
    matrix = matrix[valid_time]
    matrix[(matrix == 0.0) | ~np.isfinite(matrix)] = np.nan

    if time_axis.size == 0 or not np.isfinite(matrix).any():
        raise ValueError(f"指定范围内没有有效数据：{csv_path}")

    return time_axis, matrix


def read_well_locations(profile_name):
    csv_path = SECTION_ROOT / profile_name / "profile_wells_location.csv"
    if not csv_path.exists():
        return pd.DataFrame(columns=["wellname", "trace_index"])

    dataframe = clean_columns(pd.read_csv(csv_path))
    if not {"wellname", "trace_index"}.issubset(dataframe.columns):
        return pd.DataFrame(columns=["wellname", "trace_index"])

    dataframe["wellname"] = dataframe["wellname"].astype(str).str.strip()
    dataframe["trace_index"] = pd.to_numeric(dataframe["trace_index"], errors="coerce")
    dataframe = dataframe.loc[np.isfinite(dataframe["trace_index"])].copy()
    dataframe["trace_index"] = dataframe["trace_index"].astype(int)
    return dataframe.sort_values("trace_index").reset_index(drop=True)


def split_contiguous_indices(indices):
    indices = np.asarray(indices, dtype=int)
    if indices.size == 0:
        return []
    return np.split(indices, np.where(np.diff(indices) > 1)[0] + 1)


def read_well_log_segments(well_name, target, time_min, time_max):
    csv_path = LOG_DATA_DIR / f"{well_name}.csv"
    if not csv_path.exists():
        return []

    dataframe = clean_columns(pd.read_csv(csv_path))
    time_column = find_time_column(dataframe)
    if target not in dataframe.columns:
        return []

    log_time = pd.to_numeric(dataframe[time_column], errors="coerce").to_numpy(
        dtype=float
    )
    log_values = pd.to_numeric(dataframe[target], errors="coerce").to_numpy(dtype=float)
    valid = (
        np.isfinite(log_time)
        & np.isfinite(log_values)
        & (log_values != 0.0)
        & (log_time >= time_min)
        & (log_time <= time_max)
    )

    segments = []
    for group in split_contiguous_indices(np.where(valid)[0]):
        if group.size:
            order = np.argsort(log_time[group])
            segments.append((log_time[group][order], log_values[group][order]))
    return segments


def resolve_cmap_range(target, matrices):
    manual_range = CMAP_RANGES.get(target)
    if manual_range is not None:
        if len(manual_range) != 2:
            raise ValueError(f"CMAP_RANGES['{target}'] 必须为 None 或 (vmin, vmax)。")
        value_min, value_max = map(float, manual_range)
        if (
            not np.isfinite(value_min)
            or not np.isfinite(value_max)
            or value_min >= value_max
        ):
            raise ValueError(f"CMAP_RANGES['{target}'] 无效：{manual_range}")
        return value_min, value_max, "手动"

    finite_parts = [matrix[np.isfinite(matrix)] for matrix in matrices]
    finite_parts = [values for values in finite_parts if values.size]
    if not finite_parts:
        raise ValueError(f"{target} 在四条剖面中均无有效值。")

    all_values = np.concatenate(finite_parts)
    value_min, value_max = np.percentile(all_values, COLOR_PERCENTILES).astype(float)
    if value_max <= value_min:
        delta = max(abs(value_min), 1.0) * 1e-6
        value_min, value_max = value_min - delta, value_max + delta
    return value_min, value_max, f"{COLOR_PERCENTILES[0]:g}%–{COLOR_PERCENTILES[1]:g}%"


def draw_well_logs(ax, profile_name, target, matrix, time_axis, cmap, vmin, vmax):
    wells = read_well_locations(profile_name)
    time_min, time_max = float(np.min(time_axis)), float(np.max(time_axis))

    for _, row in wells.iterrows():
        trace_index = int(row["trace_index"])
        if not 0 <= trace_index < matrix.shape[1]:
            continue
        trace_x = trace_index + 0.5

        if OVERLAY_TRUE_LOG:
            for segment_time, segment_values in read_well_log_segments(
                row["wellname"], target, time_min, time_max
            ):
                if len(segment_time) < 2:
                    continue
                ax.fill_betweenx(
                    segment_time,
                    trace_x - LOG_COLOR_HALF_WIDTH - LOG_BLACK_BORDER_WIDTH,
                    trace_x + LOG_COLOR_HALF_WIDTH + LOG_BLACK_BORDER_WIDTH,
                    color="black",
                    edgecolor="none",
                    zorder=10,
                )
                x_edges = np.array(
                    [trace_x - LOG_COLOR_HALF_WIDTH, trace_x + LOG_COLOR_HALF_WIDTH]
                )
                cell_values = (
                    0.5 * (segment_values[:-1] + segment_values[1:])
                ).reshape(-1, 1)
                ax.pcolormesh(
                    x_edges,
                    segment_time,
                    cell_values,
                    shading="flat",
                    cmap=cmap,
                    vmin=vmin,
                    vmax=vmax,
                    zorder=11,
                )

        if SHOW_WELL_NAMES:
            ax.text(
                trace_x,
                WELL_LABEL_Y_AXES,
                str(row["wellname"]),
                transform=ax.get_xaxis_transform(),
                ha="center",
                va="bottom",
                fontsize=FONT_SIZE,
                clip_on=False,
                zorder=20,
            )


def format_colorbar(colorbar, target):
    if target in {"Ip", "Is"}:
        colorbar.ax.xaxis.set_major_formatter(
            FuncFormatter(lambda value, position: f"{value / 1000:.2f}")
        )
    else:
        colorbar.ax.xaxis.set_major_formatter(FormatStrFormatter("%.2f"))

    labels = {
        "Ip": r"P-impedance  $\times 10^6$ kg/(m$^2$ s)",
        "Is": r"S-impedance  $\times 10^6$ kg/(m$^2$ s)",
        "Dn": r"Density  $\times 10^3$ kg/m$^3$",
    }
    colorbar.set_label(labels[target], fontsize=FONT_SIZE, labelpad=3)
    colorbar.ax.tick_params(axis="x", labelsize=FONT_SIZE, pad=2)


# ------------------------------------------------------------------------------
# 5. 一次读取全部 12 个矩阵，并计算三列共享色限
# ------------------------------------------------------------------------------
plot_data = {}
for profile_name in PROFILE_NAMES:
    for target in TARGETS:
        csv_path = PRED_ROOT / profile_name / f"{target}_{PREDICTION_LABEL}.csv"
        plot_data[(profile_name, target)] = read_prediction(
            csv_path, PROFILE_TIME_RANGES.get(profile_name)
        )

column_ranges = {}
for target in TARGETS:
    matrices = [plot_data[(profile_name, target)][1] for profile_name in PROFILE_NAMES]
    column_ranges[target] = resolve_cmap_range(target, matrices)
    vmin, vmax, source = column_ranges[target]
    print(f"{target}: {source} 色标范围 = {vmin:.6g} ～ {vmax:.6g}")


# ------------------------------------------------------------------------------
# 6. 4 × 3 A4 横向绘图
# ------------------------------------------------------------------------------
figure = plt.figure(figsize=A4_LANDSCAPE_INCHES, dpi=DISPLAY_DPI)
grid = figure.add_gridspec(
    nrows=5,
    ncols=3,
    height_ratios=[1, 1, 1, 1, 0.09],
    left=0.065,
    right=0.99,
    top=0.935,
    bottom=0.09,
    hspace=ROW_HSPACE,
    wspace=0.10,
)

axes = np.empty((4, 3), dtype=object)
images = {}
cmap = plt.get_cmap(CMAP_NAME).copy()
cmap.set_bad(color="white")

for row_index, profile_name in enumerate(PROFILE_NAMES):
    for column_index, target in enumerate(TARGETS):
        ax = figure.add_subplot(grid[row_index, column_index])
        axes[row_index, column_index] = ax

        time_axis, matrix = plot_data[(profile_name, target)]
        vmin, vmax, _ = column_ranges[target]
        image = ax.imshow(
            matrix,
            aspect="auto",
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            extent=[
                0,
                matrix.shape[1],
                float(np.max(time_axis)),
                float(np.min(time_axis)),
            ],
            interpolation=INTERPOLATION,
            zorder=1,
        )
        images[target] = image

        draw_well_logs(ax, profile_name, target, matrix, time_axis, cmap, vmin, vmax)

        if row_index == 0:
            ax.set_title(COLUMN_TITLES[target], fontsize=FONT_SIZE, pad=14)

        ax.set_xlim(0, matrix.shape[1])
        ax.set_ylim(float(np.max(time_axis)), float(np.min(time_axis)))
        ax.set_xlabel("Trace", fontsize=FONT_SIZE, labelpad=2)
        ax.xaxis.set_major_locator(MaxNLocator(nbins=5, integer=True))
        ax.tick_params(axis="both", labelsize=FONT_SIZE, length=2.5, pad=2)

        if column_index == 0:
            ax.set_ylabel(TIME_LABEL, fontsize=FONT_SIZE, labelpad=TIME_LABEL_PAD)
            ax.text(
                PROFILE_LABEL_X,
                0.5,
                PROFILE_LABELS[profile_name],
                transform=ax.transAxes,
                fontsize=FONT_SIZE,
                rotation=PROFILE_LABEL_ROTATION,
                ha="right",
                va="center",
                clip_on=False,
            )
            ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
        else:
            ax.tick_params(axis="y", left=False, labelleft=False)

        for spine in ax.spines.values():
            spine.set_linewidth(0.6)

for column_index, target in enumerate(TARGETS):
    colorbar_ax = figure.add_subplot(grid[4, column_index])
    colorbar = figure.colorbar(
        images[target], cax=colorbar_ax, orientation="horizontal"
    )
    colorbar.locator = MaxNLocator(nbins=5)
    colorbar.update_ticks()
    format_colorbar(colorbar, target)

# 不使用 bbox_inches='tight'，确保输出画布保持严格的 A4 横向尺寸。
figure.savefig(OUTPUT_PNG, dpi=SAVE_DPI)
if SAVE_PDF:
    figure.savefig(OUTPUT_PDF)

print(f"已保存 PNG：{OUTPUT_PNG}")
if SAVE_PDF:
    print(f"已保存 PDF：{OUTPUT_PDF}")

if SHOW_FIGURE:
    plt.show()
else:
    plt.close(figure)

# 地震剖面

In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter, MaxNLocator


# ------------------------------------------------------------------------------
# 1. 路径与数据配置
# ------------------------------------------------------------------------------
def find_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "section").is_dir():
            return candidate
    raise FileNotFoundError("未找到项目根目录 HCT-QHead_Synthesis_AVO。")


PROJECT_ROOT = find_project_root()
SECTION_ROOT = PROJECT_ROOT / "section"
LOG_DATA_DIR = PROJECT_ROOT / "well"
OUTPUT_DIR = PROJECT_ROOT / "analysis" / "section"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PNG = OUTPUT_DIR / "Seismic_profiles_hared_cmap.png"
OUTPUT_PDF = OUTPUT_DIR / "Seismic_profiles_hared_cmap.pdf"
SAVE_PDF = True
SHOW_FIGURE = True

PROFILE_NAMES = ["profile_s1", "profile_s2", "profile_s3", "profile_s4"]
PROFILE_LABELS = {
    "profile_s1": "S1",
    "profile_s2": "S2",
    "profile_s3": "S3",
    "profile_s4": "S4",
}

# 数据文件角度与顶部显示标题分开设置，可在 COLUMN_TITLES 中独立修改。
ANGLES = [15, 30, 45]
COLUMN_TITLES = {
    15: "15°",
    30: "30°",
    45: "45°",
}

# 每条剖面的独立时间范围；设为 None 使用完整时间范围。
PROFILE_TIME_RANGES = {
    "profile_s1": (2.0, 2.8),
    "profile_s2": (2.0, 2.8),
    "profile_s3": (2.0, 2.8),
    "profile_s4": (2.0, 2.8),
}

# ------------------------------------------------------------------------------
# 2. 色标接口
# ------------------------------------------------------------------------------
COLOR_PERCENTILES = (2.0, 98.0)

# 三个角度、四条剖面共用同一个色标范围。
SHARED_CMAP_RANGE = (-4100, 4100)
CMAP_RANGES = {angle: SHARED_CMAP_RANGE for angle in ANGLES}

CMAP_NAME = "seismic"
INTERPOLATION = "bilinear"
COLORBAR_TICK_STEP = 500
COLORBAR_LABEL = r"Seismic ($\times 10^3$)"

# ------------------------------------------------------------------------------
# 3. 版式接口（与上方剖面图保持一致）
# ------------------------------------------------------------------------------
A4_LANDSCAPE_INCHES = (11.69, 8.27)
FONT_SIZE = 9  # 小五号
DISPLAY_DPI = 120
SAVE_DPI = 600
ROW_HSPACE = 0.6  # 四行剖面之间的间隔；数值越大，行间距越大
TIME_LABEL = "Time(s)"  # 每行纵轴的时间标题
TIME_LABEL_PAD = 2  # Time(s) 与时间刻度数字之间的距离
PROFILE_LABEL_ROTATION = 0  # S1-S4 水平正向显示
PROFILE_LABEL_X = -0.16  # S1-S4 的水平位置（axes 坐标，越小越靠左）

# 地震图井位标记接口：仅画单根黑线，不叠加真实测井彩条。
SHOW_WELL_NAMES = True
WELL_LINE_WIDTH = 0.8
WELL_LABEL_Y_AXES = 1.015

mpl.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Noto Sans CJK JP", "DejaVu Sans"],
        "axes.unicode_minus": False,
        "font.size": FONT_SIZE,
        "axes.titlesize": FONT_SIZE,
        "axes.labelsize": FONT_SIZE,
        "xtick.labelsize": FONT_SIZE,
        "ytick.labelsize": FONT_SIZE,
    }
)

print(f"项目根目录：{PROJECT_ROOT}")
print(f"图片输出：{OUTPUT_PNG}")

# ------------------------------------------------------------------------------
# 4. 数据读取与色标计算
# ------------------------------------------------------------------------------
def clean_columns(dataframe):
    dataframe = dataframe.copy()
    dataframe.columns = [
        str(column).replace("\ufeff", "").strip() for column in dataframe.columns
    ]
    return dataframe


def find_time_column(dataframe):
    for column in dataframe.columns:
        if str(column).lower().startswith("time"):
            return column
    raise ValueError(f"未找到时间列，当前列名：{list(dataframe.columns)}")


def read_seismic(csv_path, time_range):
    if not csv_path.exists():
        raise FileNotFoundError(f"地震数据文件不存在：{csv_path}")

    dataframe = clean_columns(pd.read_csv(csv_path))
    time_column = find_time_column(dataframe)
    time_axis = pd.to_numeric(dataframe[time_column], errors="coerce").to_numpy(
        dtype=float
    )
    data_columns = [column for column in dataframe.columns if column != time_column]
    matrix = (
        dataframe[data_columns]
        .apply(pd.to_numeric, errors="coerce")
        .to_numpy(dtype=float)
    )

    valid_time = np.isfinite(time_axis)
    if time_range is not None:
        time_start, time_end = time_range
        if time_start is not None:
            valid_time &= time_axis >= float(time_start)
        if time_end is not None:
            valid_time &= time_axis <= float(time_end)

    time_axis = time_axis[valid_time]
    matrix = matrix[valid_time]
    matrix[(matrix == 0.0) | ~np.isfinite(matrix)] = np.nan

    if time_axis.size == 0 or matrix.size == 0:
        raise ValueError(f"指定时间范围内没有有效数据：{csv_path}")
    return time_axis, matrix


def read_well_locations(profile_name):
    csv_path = SECTION_ROOT / profile_name / "profile_wells_location.csv"
    if not csv_path.exists():
        return pd.DataFrame(columns=["wellname", "trace_index"])

    dataframe = clean_columns(pd.read_csv(csv_path))
    if not {"wellname", "trace_index"}.issubset(dataframe.columns):
        return pd.DataFrame(columns=["wellname", "trace_index"])

    dataframe["wellname"] = dataframe["wellname"].astype(str).str.strip()
    dataframe["trace_index"] = pd.to_numeric(dataframe["trace_index"], errors="coerce")
    dataframe = dataframe.loc[np.isfinite(dataframe["trace_index"])].copy()
    dataframe["trace_index"] = dataframe["trace_index"].astype(int)
    return dataframe.sort_values("trace_index").reset_index(drop=True)


def read_well_max_time(well_name):
    csv_path = LOG_DATA_DIR / f"{well_name}.csv"
    if not csv_path.exists():
        return None

    dataframe = clean_columns(pd.read_csv(csv_path))
    time_column = find_time_column(dataframe)
    log_time = pd.to_numeric(dataframe[time_column], errors="coerce").to_numpy(
        dtype=float
    )
    valid_log = np.zeros(len(dataframe), dtype=bool)
    for target in ("Ip", "Is", "Dn"):
        if target not in dataframe.columns:
            continue
        values = pd.to_numeric(dataframe[target], errors="coerce").to_numpy(dtype=float)
        valid_log |= np.isfinite(values) & (values != 0.0)

    valid = np.isfinite(log_time) & valid_log
    if not np.any(valid):
        return None
    return float(np.max(log_time[valid]))


def draw_well_markers(ax, profile_name, n_traces, time_min, time_max):
    for _, row in read_well_locations(profile_name).iterrows():
        trace_index = int(row["trace_index"])
        if not 0 <= trace_index < n_traces:
            continue
        well_max_time = read_well_max_time(row["wellname"])
        if well_max_time is None or well_max_time < time_min:
            continue
        trace_x = trace_index + 0.5
        line_end_time = min(well_max_time, time_max)
        ax.plot(
            [trace_x, trace_x],
            [time_min, line_end_time],
            color="black",
            linewidth=WELL_LINE_WIDTH,
            zorder=10,
        )
        if SHOW_WELL_NAMES:
            ax.text(
                trace_x,
                WELL_LABEL_Y_AXES,
                str(row["wellname"]),
                transform=ax.get_xaxis_transform(),
                ha="center",
                va="bottom",
                fontsize=FONT_SIZE,
                clip_on=False,
                zorder=20,
            )


def resolve_color_range(angle, matrices):
    manual_range = CMAP_RANGES.get(angle)
    if manual_range is not None:
        range_start, range_end = map(float, manual_range)
        absolute_limit = max(abs(range_start), abs(range_end))
        source = "手动"
    else:
        valid_parts = [matrix[np.isfinite(matrix)] for matrix in matrices]
        valid_parts = [part for part in valid_parts if part.size]
        if not valid_parts:
            raise ValueError(f"theta{angle} 的四条剖面均无有效值。")
        combined = np.concatenate(valid_parts)
        percentile_min, percentile_max = np.percentile(combined, COLOR_PERCENTILES)
        absolute_limit = max(abs(percentile_min), abs(percentile_max))
        source = f"四剖面共同 {COLOR_PERCENTILES[0]:g}%–{COLOR_PERCENTILES[1]:g}%"

    if not np.isfinite(absolute_limit) or absolute_limit <= 0:
        raise ValueError(f"theta{angle} 的对称色标绝对值范围无效：{absolute_limit}")
    return -float(absolute_limit), float(absolute_limit), source


# ------------------------------------------------------------------------------
# 5. 读取 12 份数据，并按角度计算四剖面共同色标
# ------------------------------------------------------------------------------
seismic_data = {}
for profile_name in PROFILE_NAMES:
    for angle in ANGLES:
        csv_path = SECTION_ROOT / profile_name / f"seismic_theta{angle}.csv"
        seismic_data[(profile_name, angle)] = read_seismic(
            csv_path, PROFILE_TIME_RANGES[profile_name]
        )

color_ranges = {}
for angle in ANGLES:
    matrices = [seismic_data[(profile_name, angle)][1] for profile_name in PROFILE_NAMES]
    vmin, vmax, source = resolve_color_range(angle, matrices)
    color_ranges[angle] = (vmin, vmax)
    print(f"theta{angle}: {source} 色标范围 = {vmin:.6g} ～ {vmax:.6g}")

# ------------------------------------------------------------------------------
# 6. 4 行 × 3 列绘图
# ------------------------------------------------------------------------------
figure = plt.figure(figsize=A4_LANDSCAPE_INCHES, dpi=DISPLAY_DPI)
grid = figure.add_gridspec(
    nrows=5,
    ncols=3,
    height_ratios=[1, 1, 1, 1, 0.09],
    left=0.065,
    right=0.99,
    top=0.935,
    bottom=0.09,
    hspace=ROW_HSPACE,
    wspace=0.10,
)

cmap = mpl.colormaps[CMAP_NAME].copy()
cmap.set_bad("white")
images = {}

for row_index, profile_name in enumerate(PROFILE_NAMES):
    for column_index, angle in enumerate(ANGLES):
        ax = figure.add_subplot(grid[row_index, column_index])
        time_axis, matrix = seismic_data[(profile_name, angle)]
        vmin, vmax = color_ranges[angle]

        image = ax.imshow(
            np.ma.masked_invalid(matrix),
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            aspect="auto",
            extent=[0, matrix.shape[1], float(np.max(time_axis)), float(np.min(time_axis))],
            interpolation=INTERPOLATION,
        )
        images[angle] = image

        draw_well_markers(
            ax,
            profile_name,
            matrix.shape[1],
            float(np.min(time_axis)),
            float(np.max(time_axis)),
        )

        if row_index == 0:
            ax.set_title(COLUMN_TITLES[angle], fontsize=FONT_SIZE, pad=14)

        ax.set_xlim(0, matrix.shape[1])
        ax.set_ylim(float(np.max(time_axis)), float(np.min(time_axis)))
        ax.set_xlabel("Trace", fontsize=FONT_SIZE, labelpad=2)
        ax.xaxis.set_major_locator(MaxNLocator(nbins=5, integer=True))
        ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE, length=2.5, pad=2)

        if column_index == 0:
            ax.set_ylabel(TIME_LABEL, fontsize=FONT_SIZE, labelpad=TIME_LABEL_PAD)
            ax.text(
                PROFILE_LABEL_X,
                0.5,
                PROFILE_LABELS[profile_name],
                transform=ax.transAxes,
                fontsize=FONT_SIZE,
                rotation=PROFILE_LABEL_ROTATION,
                ha="right",
                va="center",
                clip_on=False,
            )
        else:
            ax.tick_params(axis="y", labelleft=False)

colorbar_ax = figure.add_subplot(grid[4, :])
colorbar = figure.colorbar(images[ANGLES[-1]], cax=colorbar_ax, orientation="horizontal")
shared_vmin, shared_vmax = color_ranges[ANGLES[0]]
first_tick = np.ceil(shared_vmin / COLORBAR_TICK_STEP) * COLORBAR_TICK_STEP
last_tick = np.floor(shared_vmax / COLORBAR_TICK_STEP) * COLORBAR_TICK_STEP
colorbar.set_ticks(
    np.arange(first_tick, last_tick + 0.5 * COLORBAR_TICK_STEP, COLORBAR_TICK_STEP)
)
colorbar.ax.xaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value / 1e3:g}"))
colorbar.ax.tick_params(labelsize=FONT_SIZE, length=2.5, pad=2)
colorbar.set_label(COLORBAR_LABEL, fontsize=FONT_SIZE, labelpad=2)

figure.savefig(OUTPUT_PNG, dpi=SAVE_DPI, bbox_inches=None, facecolor="white")
print(f"已保存 PNG：{OUTPUT_PNG}")

if SAVE_PDF:
    figure.savefig(OUTPUT_PDF, bbox_inches=None, facecolor="white")
    print(f"已保存 PDF：{OUTPUT_PDF}")

if SHOW_FIGURE:
    plt.show()
else:
    plt.close(figure)

# end